<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Accounts und API-Keys einrichten und testen
</b></font> </br></p>

---

Für die Notebooks im Kurs werden Konten bei **OpenAI**, **Hugging Face** und **LangSmith** benötigt. Die Schlüssel werden in Google Colab als **Secrets** hinterlegt und nie in den Notebook-Code geschrieben. Dieses Notebook zeigt, wie die Secrets angelegt werden, und prüft danach, ob alle drei Zugänge funktionieren.

**Praxis-Transfer:** Ohne gültige Zugänge kann der PMO-Assistent weder ein Modell aufrufen noch Traces in LangSmith schreiben. Ein einmaliger Test vor dem Kurs spart später die Fehlersuche.

<p><font color='black' size="5">
Secrets in Google Colab anlegen
</font></p>

1. Das Colab-Notebook öffnen.
2. In der linken Seitenleiste das **Schlüssel-Symbol** (**Secrets**) auswählen. Ist die Seitenleiste eingeklappt, zuerst auf **Weitere** klicken.
3. **Add new secret** auswählen.
4. Den Namen aus der Tabelle eintragen und den Schlüssel in das Feld **Value** kopieren.
5. Den Schalter **Notebook access** aktivieren. Ohne diese Freigabe kann das Notebook das Secret nicht lesen.
6. Die Schritte für jeden Dienst wiederholen. Die Freigabe muss für jedes Notebook einzeln erteilt werden.

| Dienst | Secret-Name | Schlüssel erstellen unter |
|---|---|---|
| OpenAI | `OPENAI_API_KEY` | platform.openai.com → **API keys** (Prepaid-Guthaben nötig) |
| Hugging Face | `HF_TOKEN` | huggingface.co → Profil → **Access Tokens** (Typ **READ**) |
| LangSmith | `LANGSMITH_API_KEY` | eu.smith.langchain.com → **Settings** → **API Keys** (EU-Workspace) |

<p><font color='darkblue' size="4">
⚠️ <b>Warnung</b>
</font></p>

API-Schlüssel sind vertrauliche Zugangsdaten. Sie gehören nicht in Markdown-Zellen, sichtbaren Code, Ausgaben, Screenshots oder GitHub-Repositories. Ein versehentlich veröffentlichter Schlüssel muss im Anbieterportal widerrufen und neu erstellt werden.

In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "false"
# Tracing bleibt standardmäßig aus; ausgewählte Referenzläufe werden explizit aktiviert.
os.environ["LANGSMITH_PROJECT"] = "m90_api-keys-test"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace,
)

# setup_api_keys(['OPENAI_API_KEY', 'HF_TOKEN', 'LANGSMITH_API_KEY'], create_globals=False)
# print()
# check_environment()
# print()
# get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import (
    BASELINE,
    ROUTER,
    JUDGE,
    PLANNER,
    WORKER,
    WORKER_PREMIUM,
    CODING,
    EMBEDDINGS,
)
# LangSmith Tracing
run_cfg = {
    "tags": ["setup", "api-keys"],
    "metadata": {"notebook": "api-keys-einrichten-und-testen", "version": "1.0"}
}

# 1 | OpenAI testen
---

Der Test ruft das Basismodell des Kurses einmal auf. `init_chat_model()` liest `OPENAI_API_KEY` aus der Umgebung; eine Antwort bedeutet, dass Schlüssel und Prepaid-Guthaben in Ordnung sind.

In [ ]:
setup_api_keys(['OPENAI_API_KEY'], create_globals=False)

In [ ]:
from langchain.chat_models import init_chat_model

try:
    llm = init_chat_model(BASELINE)   # verwendet OPENAI_API_KEY aus der Umgebung
    antwort = llm.invoke("Antworte nur mit dem Wort: Hallo")
    print("✅ OpenAI API-Key funktioniert!")
    print(f"   Antwort des Modells: {antwort.content}")
except Exception as e:
    print(f"❌ Fehler bei der Verbindung zu OpenAI: {e}")

# 2 | Hugging Face testen
---

Der Token wird benötigt, um Modelle aus dem **Hugging Face Hub** zu laden, etwa mit `transformers` oder `huggingface_hub`. Der Test fragt mit `HF_TOKEN` das zugehörige Konto ab; ungültige Token werden dabei abgelehnt.

In [ ]:
setup_api_keys(['HF_TOKEN'], create_globals=False)

In [ ]:
from huggingface_hub import whoami

try:
    konto = whoami()   # verwendet HF_TOKEN aus der Umgebung
    print(f"✅ Der Token ist gültig. Angemeldet als: {konto['name']}")
except Exception as e:
    print(f"❌ Fehler bei der Anmeldung bei Hugging Face: {e}")

# 3 | LangSmith testen
---

Der Test fragt über den EU-Endpoint (`https://eu.api.smith.langchain.com`, in der Setup-Zelle gesetzt) ein Projekt ab. Wird der Account auf der EU-Plattform angelegt, aber ein anderer Endpoint verwendet, kann der Schlüssel nicht validiert werden.

In [ ]:
setup_api_keys(['LANGSMITH_API_KEY'], create_globals=False)

In [ ]:
from langsmith import Client

try:
    client = Client()   # verwendet LANGSMITH_API_KEY und LANGSMITH_ENDPOINT aus der Umgebung
    projects = list(client.list_projects(limit=1))   # einfache Abfrage als Verbindungstest
    print("✅ LangSmith API-Key funktioniert! Verbindung erfolgreich hergestellt.")
except Exception as e:
    print(f"❌ Fehler bei der Verbindung zu LangSmith: {e}")

<p><font color='darkblue' size="4">
✅ <b>Erfolg</b>
</font></p>

Zeigen alle drei Tests ein ✅, sind die Voraussetzungen für die Kursnotebooks erfüllt. Bei einem ❌ zuerst Secret-Name, Schalter **Notebook access** und Schlüssel prüfen und danach die Setup-Zelle erneut ausführen.